# Image Segmentation

## Types of Segmentation

| Type | Output | Description |
|------|--------|-------------|
| **Semantic** | Per-pixel class label | Every pixel gets a class, no instance separation |
| **Instance** | Per-object mask | Separate mask per object instance |
| **Panoptic** | Semantic + Instance | Unified: things (countable) + stuff (background) |

---

## Evaluation Metrics

**Pixel Accuracy**:
$$PA = \frac{\sum_i n_{ii}}{\sum_i t_i}$$

**Mean Intersection over Union (mIoU)**:
$$mIoU = \frac{1}{k+1}\sum_{i=0}^{k} \frac{p_{ii}}{\sum_j p_{ij} + \sum_j p_{ji} - p_{ii}}$$

where $p_{ij}$ = number of pixels of class $i$ predicted as class $j$.

**Dice Coefficient** (F1 for segmentation):
$$\text{Dice} = \frac{2|A \cap B|}{|A| + |B|} = \frac{2TP}{2TP + FP + FN}$$

**Dice Loss** (differentiable):
$$\mathcal{L}_{Dice} = 1 - \frac{2\sum_i p_i g_i}{\sum_i p_i^2 + \sum_i g_i^2 + \epsilon}$$

---

## Key Architectures

### FCN (Fully Convolutional Network, 2015)
First end-to-end trainable segmentation network. Replaces FC layers with convolutions. Uses skip connections from lower layers.

### U-Net (2015)
Encoder-decoder with **skip connections** at each resolution level:
- Encoder: contracting path (conv + maxpool), captures context
- Decoder: expansive path (upsampling + conv), recovers resolution
- Skip connections: concatenate encoder features to decoder
- Originally designed for biomedical image segmentation

U-Net++ improves on U-Net with nested dense skip connections.

### DeepLab Family

**Atrous (Dilated) Convolution** increases receptive field without losing resolution:
$$y[i] = \sum_k x[i + r \cdot k] \cdot w[k]$$

where $r$ = dilation rate. Rate $r=1$ = standard conv, $r=2$ = skip one pixel.

**ASPP (Atrous Spatial Pyramid Pooling)** applies atrous conv at multiple rates to capture multi-scale context.

| Version | Year | Key Addition |
|---------|------|-------------|
| DeepLabv1 | 2014 | Atrous conv + CRF |
| DeepLabv2 | 2016 | ASPP |
| DeepLabv3 | 2017 | Improved ASPP |
| DeepLabv3+ | 2018 | Encoder-decoder with ASPP |

### Mask R-CNN (2017)
Extends Faster R-CNN with a mask head:
- RPN → RoIAlign → Box head + Mask head
- Mask head predicts $K$ binary masks (one per class) for each RoI
- **RoIAlign** fixes quantization artifacts using bilinear interpolation:
$$f(x,y) = \sum_{i,j} \max(0, 1-|x-x_i|) \cdot \max(0, 1-|y-y_j|) \cdot f(x_i, y_j)$$

### SegFormer (2021)
- Hierarchical Transformer encoder (Mix Transformer / MiT)
- No positional encoding → can handle any resolution
- Lightweight All-MLP decoder

### SAM Segment Anything Model (2023)
Foundation model for segmentation:
- Trained on SA-1B (1B masks, 11M images)
- Promptable: point, box, text, or mask prompts
- Architecture: image encoder (MAE ViT-H) + prompt encoder + mask decoder
- SAM2 (2024): extends to video segmentation with memory bank

---

## Loss Functions for Segmentation

**Cross-Entropy Loss** (most common):
$$\mathcal{L}_{CE} = -\sum_i \sum_c y_{ic} \log(\hat{p}_{ic})$$

**Dice Loss** (handles class imbalance):
$$\mathcal{L}_{Dice} = 1 - \frac{2\sum p \cdot g}{\sum p^2 + \sum g^2 + \epsilon}$$

**Tversky Loss** (generalized Dice, control FP/FN balance):
$$\mathcal{L}_{Tversky} = 1 - \frac{TP}{TP + \alpha \cdot FP + \beta \cdot FN}$$

**Combined Loss**:
$$\mathcal{L} = \mathcal{L}_{CE} + \mathcal{L}_{Dice}$$


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# ============================================================
# U-Net from Scratch
# ============================================================

class DoubleConv(nn.Module):
    """Two consecutive Conv-BN-ReLU blocks."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2, features=[64,128,256,512]):
        super().__init__()
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()
        self.pool    = nn.MaxPool2d(2, 2)

        # Encoder (contracting path)
        ch = in_channels
        for f in features:
            self.encoder.append(DoubleConv(ch, f))
            ch = f

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)

        # Decoder (expansive path)
        for f in reversed(features):
            self.decoder.append(nn.ConvTranspose2d(f*2, f, kernel_size=2, stride=2))
            self.decoder.append(DoubleConv(f*2, f))  # f*2 because of skip concat

        # Final 1x1 conv
        self.final_conv = nn.Conv2d(features[0], num_classes, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        # Encoding
        for enc in self.encoder:
            x = enc(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]  # Reverse

        # Decoding
        for i in range(0, len(self.decoder), 2):
            x = self.decoder[i](x)          # Upsample
            skip = skip_connections[i//2]

            # Handle size mismatch
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])

            x = torch.cat([skip, x], dim=1) # Skip connection
            x = self.decoder[i+1](x)        # Double conv

        return self.final_conv(x)


# Test
model = UNet(in_channels=3, num_classes=21)  # 21 classes like PASCAL VOC
x = torch.randn(2, 3, 256, 256)
out = model(x)
print(f'Input:  {x.shape}')
print(f'Output: {out.shape}')  # Should be (2, 21, 256, 256)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

Input:  torch.Size([2, 3, 256, 256])
Output: torch.Size([2, 21, 256, 256])
Total parameters: 31,038,933


In [3]:
# ============================================================
# Loss Functions for Segmentation
# ============================================================

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred = torch.sigmoid(pred)
        pred   = pred.contiguous().view(-1)
        target = target.contiguous().view(-1).float()
        intersection = (pred * target).sum()
        dice = (2. * intersection + self.smooth) / (pred.sum() + target.sum() + self.smooth)
        return 1 - dice


class TverskyLoss(nn.Module):
    """Tversky loss: alpha controls FP, beta controls FN penalty."""
    def __init__(self, alpha=0.5, beta=0.5, smooth=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.smooth = smooth

    def forward(self, pred, target):
        pred   = torch.sigmoid(pred).view(-1)
        target = target.view(-1).float()
        TP = (pred * target).sum()
        FP = ((1 - target) * pred).sum()
        FN = (target * (1 - pred)).sum()
        tversky = (TP + self.smooth) / (TP + self.alpha*FP + self.beta*FN + self.smooth)
        return 1 - tversky


class CombinedLoss(nn.Module):
    def __init__(self, ce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.ce_weight   = ce_weight
        self.dice_weight = dice_weight
        self.ce   = nn.CrossEntropyLoss()
        self.dice = DiceLoss()

    def forward(self, pred, target):
        ce_loss   = self.ce(pred, target)
        dice_loss = self.dice(pred[:, 1], (target == 1).float())  # binary dice
        return self.ce_weight * ce_loss + self.dice_weight * dice_loss


# Test losses
pred_logits = torch.randn(2, 2, 64, 64)
target_mask = torch.randint(0, 2, (2, 64, 64))

dice = DiceLoss()(pred_logits[:, 1], target_mask.float())
print(f'Dice Loss:    {dice.item():.4f}')

tversky = TverskyLoss(alpha=0.3, beta=0.7)(pred_logits[:,1], target_mask.float())
print(f'Tversky Loss: {tversky.item():.4f} (higher beta=0.7 penalizes FN more)')

Dice Loss:    0.4990
Tversky Loss: 0.5004 (higher beta=0.7 penalizes FN more)


In [4]:
# ============================================================
# IoU Metric Calculation
# ============================================================

def compute_miou(pred_mask, true_mask, num_classes):
    """Compute mean IoU over all classes."""
    iou_list = []
    for cls in range(num_classes):
        pred_c = (pred_mask == cls)
        true_c = (true_mask == cls)
        intersection = (pred_c & true_c).sum().float()
        union = (pred_c | true_c).sum().float()
        if union == 0:
            continue  # Skip absent classes
        iou_list.append((intersection / union).item())
    return np.mean(iou_list) if iou_list else 0.0


# Simulate predictions
num_classes = 21
H, W = 256, 256

# Perfect prediction
true_mask = torch.randint(0, num_classes, (H, W))
pred_perfect = true_mask.clone()
print(f'Perfect mIoU: {compute_miou(pred_perfect, true_mask, num_classes):.4f}')

# Random prediction
pred_random = torch.randint(0, num_classes, (H, W))
print(f'Random mIoU:  {compute_miou(pred_random, true_mask, num_classes):.4f}')

Perfect mIoU: 1.0000
Random mIoU:  0.0245


In [5]:
# ============================================================
# SAM Segment Anything Model Usage
# ============================================================
# pip install segment-anything
# pip install git+https://github.com/facebookresearch/segment-anything.git

sam_code = '''
import torch
from segment_anything import SamAutomaticMaskGenerator, SamPredictor, sam_model_registry

# Load model (vit_h is largest, vit_b is smallest)
sam = sam_model_registry["vit_h"](checkpoint="sam_vit_h_4b8939.pth")
sam.to("cuda")

# --- Automatic mask generation (segment everything) ---
mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,
    pred_iou_thresh=0.86,
    stability_score_thresh=0.92,
)
masks = mask_generator.generate(image)  # image: numpy HxWx3 uint8
# Returns list of dicts: {segmentation, area, bbox, predicted_iou, ...}

# --- Interactive segmentation with prompts ---
predictor = SamPredictor(sam)
predictor.set_image(image)

# Point prompt
point_coords = np.array([[500, 375]])  # (x, y)
point_labels = np.array([1])           # 1=foreground, 0=background
masks, scores, logits = predictor.predict(
    point_coords=point_coords,
    point_labels=point_labels,
    multimask_output=True,
)

# Box prompt
box = np.array([100, 100, 400, 400])   # [x1, y1, x2, y2]
masks, _, _ = predictor.predict(box=box, multimask_output=False)

# SAM2 (for video)
# pip install git+https://github.com/facebookresearch/sam2.git
from sam2.build_sam import build_sam2_video_predictor
predictor = build_sam2_video_predictor("sam2_hiera_large.yaml", "sam2.1_hiera_large.pt")
'''
print('SAM usage code:')
print(sam_code)

SAM usage code:

import torch
from segment_anything import SamAutomaticMaskGenerator, SamPredictor, sam_model_registry

# Load model (vit_h is largest, vit_b is smallest)
sam = sam_model_registry["vit_h"](checkpoint="sam_vit_h_4b8939.pth")
sam.to("cuda")

# --- Automatic mask generation (segment everything) ---
mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,
    pred_iou_thresh=0.86,
    stability_score_thresh=0.92,
)
masks = mask_generator.generate(image)  # image: numpy HxWx3 uint8
# Returns list of dicts: {segmentation, area, bbox, predicted_iou, ...}

# --- Interactive segmentation with prompts ---
predictor = SamPredictor(sam)
predictor.set_image(image)

# Point prompt
point_coords = np.array([[500, 375]])  # (x, y)
point_labels = np.array([1])           # 1=foreground, 0=background
masks, scores, logits = predictor.predict(
    point_coords=point_coords,
    point_labels=point_labels,
    multimask_output=True,
)

# Box prompt
box = np.array([10

In [6]:
# ============================================================
# Using torchvision segmentation models
# ============================================================
from torchvision import models

# DeepLabV3 with ResNet-101 backbone
deeplabv3 = models.segmentation.deeplabv3_resnet101(
    weights=models.segmentation.DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1
)
print('DeepLabV3-ResNet101 loaded')

# FCN
fcn = models.segmentation.fcn_resnet50(
    weights=models.segmentation.FCN_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
)
print('FCN-ResNet50 loaded')

# LRASPP (lightweight for mobile)
lraspp = models.segmentation.lraspp_mobilenet_v3_large(
    weights=models.segmentation.LRASPP_MobileNet_V3_Large_Weights.COCO_WITH_VOC_LABELS_V1
)
print('LRASPP-MobileNetV3 loaded')

# Count params
for name, m in [('DeepLabV3', deeplabv3), ('FCN', fcn), ('LRASPP', lraspp)]:
    params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f'  {name}: {params:.1f}M parameters')

Downloading: "https://download.pytorch.org/models/deeplabv3_resnet101_coco-586e9e4e.pth" to /home/dell/.cache/torch/hub/checkpoints/deeplabv3_resnet101_coco-586e9e4e.pth


  0%|          | 0.00/233M [00:00<?, ?B/s]

  0%|          | 256k/233M [00:00<02:06, 1.93MB/s]

  0%|          | 896k/233M [00:00<01:01, 3.99MB/s]

  1%|          | 1.88M/233M [00:00<00:37, 6.53MB/s]

...

 75%|███████▌  | 9.38M/12.5M [00:01<00:00, 10.7MB/s]

 84%|████████▍ | 10.5M/12.5M [00:01<00:00, 7.81MB/s]

 99%|█████████▉| 12.4M/12.5M [00:01<00:00, 10.5MB/s]

100%|██████████| 12.5M/12.5M [00:01<00:00, 8.38MB/s]

LRASPP-MobileNetV3 loaded
  DeepLabV3: 61.0M parameters
  FCN: 35.3M parameters
  LRASPP: 3.2M parameters


## Additional Learning Resources

### Papers
- [FCN: Fully Convolutional Networks for Semantic Segmentation](https://arxiv.org/abs/1411.4038) Long et al. 2015
- [U-Net](https://arxiv.org/abs/1505.04597) Ronneberger et al. 2015
- [DeepLabv3+](https://arxiv.org/abs/1802.02611) Chen et al. 2018
- [Mask R-CNN](https://arxiv.org/abs/1703.06870) He et al. 2017
- [SegFormer](https://arxiv.org/abs/2105.15203) Xie et al. 2021
- [SAM: Segment Anything](https://arxiv.org/abs/2304.02643) Kirillov et al. 2023
- [SAM2: Segment Anything in Images and Videos](https://arxiv.org/abs/2408.00714) Ravi et al. 2024
- [Panoptic Segmentation](https://arxiv.org/abs/1801.00868) Kirillov et al. 2019
- [Tversky Loss](https://arxiv.org/abs/1706.05721) Salehi et al. 2017

### Frameworks
- [Detectron2](https://github.com/facebookresearch/detectron2) Mask R-CNN, Panoptic FPN
- [MMSegmentation](https://mmsegmentation.readthedocs.io/) OpenMMLab segmentation toolbox
- [torchvision segmentation](https://pytorch.org/vision/stable/models.html#semantic-segmentation)
- [Segment Anything (SAM)](https://github.com/facebookresearch/segment-anything)
- [SAM2](https://github.com/facebookresearch/sam2)

### Benchmarks
- [Papers with Code ADE20K Semantic Segmentation](https://paperswithcode.com/sota/semantic-segmentation-on-ade20k)
- [COCO Panoptic Segmentation](https://paperswithcode.com/sota/panoptic-segmentation-on-coco-panoptic)
